<a href="https://colab.research.google.com/github/karthik-B17/Data-Engineering/blob/main/Mini%20Project-Analysis%20on%20Ecommerce%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("CustomerDataProcessing").getOrCreate()

In [ ]:
spark.sql('show tables').show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [ ]:
df=spark.read.format('csv').option('header','true').option('inferSchema','true').load("/content/My DataSets/customers_10mb.csv")

In [ ]:
df.show(10)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     True|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    False|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     True|
|          3|Customer_3|Bangalore|  Telangana|  India|       2023-03-24|     True|
|          4|Customer_4|Hyderabad|  Telangana|  India|       2023-06-04|    False|
|          5|Customer_5|Hyderabad|West Bengal|  India|       2023-07-26|     True|
|          6|Customer_6|Hyderabad|  Karnataka|  India|       2023-08-07|    False|
|          7|Customer_7|Bangalore|  Telangana|  India|       2023-08-25|     True|
|          8|Customer_8|Bangalore|Maharashtra|  India|       2023-07-13|    False|
|   

In [ ]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



In [ ]:
# spark.stop()
from pyspark.sql.functions import *


In [ ]:
df=df.withColumn('registration_date',to_date(col('registration_date'),'yyyy-MM-dd')).withColumn('is_active',col('is_active').cast('boolean'))
df.show(10)
df.printSchema()

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     true|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    false|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     true|
|          3|Customer_3|Bangalore|  Telangana|  India|       2023-03-24|     true|
|          4|Customer_4|Hyderabad|  Telangana|  India|       2023-06-04|    false|
|          5|Customer_5|Hyderabad|West Bengal|  India|       2023-07-26|     true|
|          6|Customer_6|Hyderabad|  Karnataka|  India|       2023-08-07|    false|
|          7|Customer_7|Bangalore|  Telangana|  India|       2023-08-25|     true|
|          8|Customer_8|Bangalore|Maharashtra|  India|       2023-07-13|    false|
|   

In [ ]:
df=df.fillna({'city':'Unknown','state':'Unknown','country':'Unknown'})

In [ ]:

df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     true|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    false|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     true|
|          3|Customer_3|Bangalore|  Telangana|  India|       2023-03-24|     true|
|          4|Customer_4|Hyderabad|  Telangana|  India|       2023-06-04|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows


In [ ]:
df

DataFrame[customer_id: string, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean]

In [ ]:
df=df.withColumn('registration_year',year(col('registration_date')))\
  .withColumn('registration_month',month(col('registration_date')))

In [ ]:
df.show(7)

+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|customer_id|      name|     city|      state|country|registration_date|is_active|registration_year|registration_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     true|             2023|                10|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    false|             2023|                10|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     true|             2023|                 2|
|          3|Customer_3|Bangalore|  Telangana|  India|       2023-03-24|     true|             2023|                 3|
|          4|Customer_4|Hyderabad|  Telangana|  India|       2023-06-04|    false|             2023|                 6|
|          5|Customer_5|Hyderabad|West B

In [ ]:
unique_cities=df.select('city').distinct()
unique_cities.show(5)

+---------+
|     city|
+---------+
|Bangalore|
|  Chennai|
|   Mumbai|
|Ahmedabad|
|  Kolkata|
+---------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import countDistinct

In [ ]:
unique_city_count=df.select(countDistinct('city')).collect()[0][0]
print(unique_city_count)

8


In [ ]:
spark

In [ ]:
unique_state_count=df.select(countDistinct('state')).collect()[0][0]
print(unique_state_count)

7


In [ ]:
df.groupBy('city','country').count().orderBy(col('count').desc()).show()

+---------+-------+-----+
|     city|country|count|
+---------+-------+-----+
|     Pune|  India|21481|
|Ahmedabad|  India|21272|
|Bangalore|  India|21272|
|  Kolkata|  India|21264|
|Hyderabad|  India|21174|
|    Delhi|  India|21123|
|  Chennai|  India|21046|
|   Mumbai|  India|21041|
+---------+-------+-----+



In [ ]:
df.groupBy('city').pivot('is_active').count().show()

+---------+-----+-----+
|     city|false| true|
+---------+-----+-----+
|Bangalore|10615|10657|
|  Chennai|10570|10476|
|   Mumbai|10323|10718|
|Ahmedabad|10576|10696|
|  Kolkata|10648|10616|
|     Pune|10707|10774|
|    Delhi|10504|10619|
|Hyderabad|10524|10650|
+---------+-----+-----+



In [ ]:

from pyspark.sql.functions import col

In [ ]:
window_specs=Window.partitionBy('state').orderBy(col('registration_date').desc())

In [ ]:
window_specs

In [ ]:
df.withColumn('rank',rank().over(window_specs))\
.withColumn('dense_rank',dense_rank().over(window_specs))\
.withColumn('row_number',row_number().over(window_specs)).show(10)

+-----------+--------------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+
|customer_id|          name|     city|state|country|registration_date|is_active|registration_year|registration_month|rank|dense_rank|row_number|
+-----------+--------------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+
|       3674| Customer_3674|Bangalore|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         1|
|       3925| Customer_3925|Bangalore|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         2|
|       6885| Customer_6885|   Mumbai|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         3|
|       7465| Customer_7465|    Delhi|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|        

In [ ]:
df_recent_cust=df.filter(col('registration_date')>= lit('2023-07-01'))
df_recent_cust.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|customer_id|      name|     city|      state|country|registration_date|is_active|registration_year|registration_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     true|             2023|                10|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    false|             2023|                10|
|          5|Customer_5|Hyderabad|West Bengal|  India|       2023-07-26|     true|             2023|                 7|
|          6|Customer_6|Hyderabad|  Karnataka|  India|       2023-08-07|    false|             2023|                 8|
|          7|Customer_7|Bangalore|  Telangana|  India|       2023-08-25|     true|             2023|                 8|
+-----------+----------+---------+------

In [ ]:
# Let us get the oldest and the newest customer per city

In [ ]:
df.groupBy('city').agg(min(col('registration_date')).alias('oldest'),max(col('registration_date')).alias('newest')).show()


+---------+----------+----------+
|     city|    oldest|    newest|
+---------+----------+----------+
|Bangalore|2023-01-01|2023-12-31|
|  Chennai|2023-01-01|2023-12-31|
|   Mumbai|2023-01-01|2023-12-31|
|Ahmedabad|2023-01-01|2023-12-31|
|  Kolkata|2023-01-01|2023-12-31|
|     Pune|2023-01-01|2023-12-31|
|    Delhi|2023-01-01|2023-12-31|
|Hyderabad|2023-01-01|2023-12-31|
+---------+----------+----------+



In [ ]:
output_path="/content/spark-warehouse/processed_customers"
df.write.mode('overwrite').format('parquet').save(output_path)

In [ ]:
! ls spark-warehouse/

processed_customers


In [ ]:
df.write.mode('overwrite').format('parquet').saveAsTable('Customer_Analysis')

In [ ]:
spark.sql('show tables').show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|  default|customer_analysis|      false|
+---------+-----------------+-----------+



In [ ]:
spark.sql('select * from customer_analysis limit 5').show()

+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|customer_id|      name|     city|      state|country|registration_date|is_active|registration_year|registration_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     true|             2023|                10|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    false|             2023|                10|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     true|             2023|                 2|
|          3|Customer_3|Bangalore|  Telangana|  India|       2023-03-24|     true|             2023|                 3|
|          4|Customer_4|Hyderabad|  Telangana|  India|       2023-06-04|    false|             2023|                 6|
+-----------+----------+---------+------

# Joining And Analyzing customers and Orders

In [ ]:
df_orders=spark.read.format('csv').option('header','true').option('inferSchema','true').csv('/content/My DataSets/orders_10mb.csv')

In [ ]:
df_orders.count()

169673

In [ ]:
df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- status: string (nullable = true)



In [ ]:
df_orders.dropna(subset=['order_id','status'])

DataFrame[order_id: int, customer_id: int, order_date: date, total_amount: double, status: string]

In [ ]:
# Analysis on Orders

In [ ]:
# df_orders = df_orders.dropna(subset=['order_id', 'customer_id'])
# No Null entries here Already checked


In [ ]:
mode_date=df_orders.select(mode(col('order_date'))).collect()[0][0]

In [ ]:
mode_date=str(mode_date)
print(mode_date)

2024-02-17


In [226]:
average_amt =df_orders.select(round(mean(col('total_amount')))).collect()[0][0]


In [227]:
df_orders=df_orders.fillna({'order_date':mode_date,'total_amount':average_amt})

In [231]:
customer_orders_df=df.join(df_orders,'customer_id',"inner")

In [232]:
customer_orders_df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+--------+----------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|order_id|order_date|     total_amount|   status|
+-----------+----------+---------+-----------+-------+-----------------+---------+--------+----------+-----------------+---------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     true|    2552|2024-06-20|947.8474347297807|  Shipped|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    false|  144006|2024-09-01|945.1783810475038|Cancelled|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     true|  136245|2024-01-18|995.5799727041127|Cancelled|
|          5|Customer_5|Hyderabad|West Bengal|  India|       2023-07-26|     true|   75022|2024-01-04| 768.167470468336|  Shipped|
|          6|Customer_6|Hyderabad|  Karnataka|  India|       2023-08-07|    false| 

In [230]:
# Total orders per customers

In [244]:
customer_order_count=customer_orders_df.groupBy('customer_id').agg(count('*').alias('cnt'),sum(col('total_amount')).alias('amt')).orderBy(col('amt').desc(),col('cnt').desc())
customer_order_count.show()

+-----------+---+------------------+
|customer_id|cnt|               amt|
+-----------+---+------------------+
|       4680|  7| 5138.631065963803|
|     144160|  6| 4590.783328014365|
|       1822|  8| 4587.463046775093|
|     119990|  6| 4584.205837473942|
|     132046|  6| 4373.860921890816|
|     116459|  8| 4296.914086806486|
|      19141|  7| 4244.870486493495|
|      72794|  6| 4188.187293249394|
|       3255|  5|  4183.03741171044|
|     150927|  6| 4163.655455957863|
|      61531|  6| 4163.381512985349|
|      27584|  5| 4157.363677138945|
|     122760|  5| 4103.596837084001|
|     160283|  5|4081.5235548997844|
|      72820|  6| 4056.009473661086|
|     134539|  5|4050.6699582577344|
|      73396|  8|4025.7348357282794|
|      96399|  5| 4022.883419101303|
|      13490|  5|4013.7088833125717|
|     140447|  6|3977.5008005095724|
+-----------+---+------------------+
only showing top 20 rows


In [245]:
# Order By status

In [246]:
orders_status_count=customer_orders_df.groupBy('status').count()
orders_status_count.show()

+---------+-----+
|   status|count|
+---------+-----+
|  Shipped|42395|
|Cancelled|42606|
|Delivered|42626|
|  Pending|42046|
+---------+-----+



In [247]:
# order By month

In [249]:
orders_by_month=customer_orders_df.groupBy(month(col('order_date')).alias('month')).count().orderBy('month')
orders_by_month.show()

+-----+-----+
|month|count|
+-----+-----+
|    1|14406|
|    2|13553|
|    3|14591|
|    4|13858|
|    5|14233|
|    6|13990|
|    7|14380|
|    8|14401|
|    9|14094|
|   10|14317|
|   11|14019|
|   12|13831|
+-----+-----+



In [251]:
window_spec=Window.orderBy(col('amt').desc())

In [252]:
ranked_customers=customer_order_count.withColumn('dense_rank',dense_rank().over(window_spec))
ranked_customers.show(10)

+-----------+---+-----------------+----------+
|customer_id|cnt|              amt|dense_rank|
+-----------+---+-----------------+----------+
|       4680|  7|5138.631065963803|         1|
|     144160|  6|4590.783328014365|         2|
|       1822|  8|4587.463046775093|         3|
|     119990|  6|4584.205837473942|         4|
|     132046|  6|4373.860921890816|         5|
|     116459|  8|4296.914086806486|         6|
|      19141|  7|4244.870486493495|         7|
|      72794|  6|4188.187293249394|         8|
|       3255|  5| 4183.03741171044|         9|
|     150927|  6|4163.655455957863|        10|
+-----------+---+-----------------+----------+
only showing top 10 rows


In [253]:
!du -sh /content/*

35M	/content/mnist_train_small.csv
19M	/content/My DataSets
20M	/content/sample_data
4.0M	/content/spark-warehouse


In [254]:
!pip cache purge

Files removed: 0


In [256]:
# Finding the customers with high order frequency but low total spend

In [262]:
customer_order_count=customer_orders_df.groupBy('customer_id').agg(count('*').alias('cnt'),sum(col('total_amount')).alias('amt')).orderBy(col('cnt').desc(),col('amt').asc())
customer_order_count.show()

+-----------+---+------------------+
|customer_id|cnt|               amt|
+-----------+---+------------------+
|      73396|  8|4025.7348357282794|
|     116459|  8| 4296.914086806486|
|       1822|  8| 4587.463046775093|
|      65947|  7| 1577.037152454686|
|      75889|  7|2062.5134415692582|
|      90715|  7| 2374.565831921064|
|     169618|  7| 2966.311380282747|
|     131001|  7| 3327.101288885349|
|     162041|  7|3617.4155427073683|
|      21933|  7|3758.9482630523303|
|      19141|  7| 4244.870486493495|
|       4680|  7| 5138.631065963803|
|       1261|  6|1492.7388078366903|
|     147205|  6|1515.8001055361738|
|      78050|  6| 1576.549745630416|
|     143945|  6|1598.7490521349798|
|     139851|  6| 1809.689323366901|
|      38877|  6| 1924.588834129583|
|      79603|  6|1983.3456389797009|
|      55354|  6|2012.9305971726592|
+-----------+---+------------------+
only showing top 20 rows


In [267]:
output_path='/content/My DataSets/final_customer_orders'
customer_orders_df.write.mode('overwrite').parquet(output_path)